<a href="https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
Lane 2: Refresh / Content Opportunity Scoring

I am framing this lane mainly as a ranking/scoring task. The goal is to give each content page a score that helps rank pages for human review. A higher score means the page should be considered earlier in the review queue.

The task is not simply to classify a page as good or bad. The useful output is an ordered list of pages so a limited review team can decide which pages to inspect first for refresh, expansion, protection, pruning, or monitoring.

In [12]:

task_type = "ranking / scoring"

print("Task type:", task_type)
print("Output: a priority score and ranked review queue for content pages")

Task type: ranking / scoring
Output: a priority score and ranked review queue for content pages


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter dataset provides an observed **`trend_direction`** field. For an initial supervised experiment, I can use a defined proxy target:

**Proxy target:** `is_declining_label = (trend_direction == "down")`.

This proxy represents whether a page is currently observed as declining in the available window. It is useful for learning the framing, but it is not the same as a future causal outcome. A stronger capstone target would ideally be a future observed outcome, such as whether a page's performance declines in a later period.

The final output would be a page-level priority score/ranking, which can support a human review queue.


In [13]:
import pandas as pd
from pathlib import Path


data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    !git clone -q https://github.com/Anshikaag-28/Anshika-FlyRank-ML.git /content/Anshika-FlyRank-ML
    %cd /content/Anshika-FlyRank-ML
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Proxy target: 1 = trend_direction is 'down', 0 = otherwise")
print(
    "Positive/proxy rate:",
    round(df["is_declining_label"].mean() * 100, 1),
    "%"
)


Proxy target: 1 = trend_direction is 'down', 0 = otherwise
Positive/proxy rate: 54.2 %


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My main success metric is Precision@K, where K is the number of pages the team can realistically review first.

For example, Precision@50 asks: out of the 50 highest-ranked pages, how many are actually in the positive/proxy target group?

This metric fits the decision because the team has limited review capacity. I care about whether the top of the ranking contains useful pages, rather than only whether the model gets every page correct.

In [14]:
# Precision@50 is the main ranking metric for this task.
# These are the starter-model results from the earlier notebook work.

hand_rule_precision_at_50 = 0.240
starter_rf_precision_at_50 = 0.740

print(
    f"Hand-written rule Precision@50: "
    f"{hand_rule_precision_at_50:.3f}"
)

print(
    f"Starter random-forest Precision@50: "
    f"{starter_rf_precision_at_50:.3f}"
)

print(
    f"Approximate lift: "
    f"{starter_rf_precision_at_50 / hand_rule_precision_at_50:.1f}x"
)

Hand-written rule Precision@50: 0.240
Starter random-forest Precision@50: 0.740
Approximate lift: 3.1x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


**Unit of analysis: one content page.** Each row represents one anonymized content page and its observed search, traffic, engagement, freshness, and content attributes.

The dataframe below shows a small lane slice and creates the proxy target used above.


In [16]:

lane_columns = [
    "content_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "freshness_days",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate",
    "trend_direction",
    "is_declining_label"
]

available_columns = [
    c for c in lane_columns
    if c in df.columns
]

lane_df = df[available_columns].copy()

print("Unit of analysis: one anonymized content page")
print("Rows in lane slice:", len(lane_df))

lane_df.head(10)

Unit of analysis: one anonymized content page
Rows in lane slice: 30000


,content_id,impressions_90d,sessions_90d,content_age_days,ctr,avg_position,word_count,engagement_rate,trend_direction,is_declining_label
0,content_304f48230142,3803,17,187,0.76,10.6,3221.0,5.88,down,1
1,content_a1fb4e703a9e,15320,9,445,0.05,20.3,2481.0,0.00,down,1
2,content_9aa793d4d895,12581,11,141,0.09,36.5,3515.0,0.00,down,1
3,content_331d6c4de07b,11751,78,463,0.49,6.2,NaN,1.28,stable,0
4,content_d99b7a2d90ca,19140,145,263,0.13,44.0,2803.0,0.00,down,1
5,content_d4084a4bc775,3970,5,147,0.03,8.5,3080.0,0.00,down,1
6,content_9a34b442b552,20,1,90,0.00,7.0,3059.0,0.00,down,1
7,content_a63219c6e95a,1724,28,445,0.06,21.2,NaN,3.57,stable,0
8,content_5e6c160719bc,32574,68,90,0.09,46.0,3807.0,5.88,down,1
9,content_c27558df2b0c,1240,3,257,0.16,4.9,NaN,0.00,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule might say something like: **review every page with low CTR** or **review every page whose trend is down**. That is easy to understand, but it uses one signal or a manually chosen threshold.

The page-review problem is more complicated because several observable signals can interact: impressions, clicks, sessions, average position, CTR, content age, freshness, word count, and engagement. A page can have a high impression volume but weak CTR, or a strong position but declining performance, so one threshold may miss useful combinations.

ML can learn a ranking/scoring pattern from multiple signals and produce a consistent priority order. The value is therefore in **decision support and ranking quality**, not in claiming that the model knows Google's ranking algorithm or that a recommended refresh will definitely improve traffic.


In [17]:

assert "content_id" in df.columns
assert "is_declining_label" in df.columns
assert df["content_id"].notna().all()

print("Unit check passed: each displayed record is a content page.")
print("Target check passed: is_declining_label is available as a defined proxy.")


Unit check passed: each displayed record is a content page.
Target check passed: is_declining_label is available as a defined proxy.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.